<a href="https://colab.research.google.com/github/Sutharsan2006/GenAI/blob/GenAI-lab/Ex_No_6_RETRIEVAL_AUGMENTED_GENERATION_(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# --------------------------------------------------
# 1. Knowledge Base
# --------------------------------------------------

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# --------------------------------------------------
# 2. Load Embedding Model
# --------------------------------------------------

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True
)

# --------------------------------------------------
# 3. Create FAISS Index
# --------------------------------------------------

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings)

# --------------------------------------------------
# 4. User Query
# --------------------------------------------------

query = "What is RAG in AI?"

# Convert query into embedding
query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True
)

# Search top 2 relevant documents
D, I = index.search(query_embedding, k=2)

retrieved_chunks = [
    documents[i]
    for i in I[0]
]

# --------------------------------------------------
# 5. Create Context
# --------------------------------------------------

context = " ".join(retrieved_chunks)

prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

# --------------------------------------------------
# 6. Load FLAN-T5 Directly
# --------------------------------------------------

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# --------------------------------------------------
# 7. Generate Answer
# --------------------------------------------------

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = model.generate(
    **inputs,
    max_new_tokens=60
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# --------------------------------------------------
# 8. Display Results
# --------------------------------------------------

print("Retrieved Context:")

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}")

print("\nQuestion:")
print(query)

print("\nAnswer:")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Retrieved Context:
1. Python is a popular high-level programming language used in AI development.
2. Retrieval-Augmented Generation combines document retrieval with text generation.

Question:
What is RAG in AI?

Answer:
combines document retrieval with text generation
